In [ ]:
import sys
import os

# 1. Determinar la ruta absoluta de la carpeta raíz (un nivel por encima de 'notebooks')
ruta_raiz = os.path.abspath(os.path.join('..'))

# 2. Añadir la ruta raíz a las variables de entorno para importar módulos 'src'
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

# 3. Cambiar el directorio de trabajo del cuaderno a la raíz del proyecto para leer 'data/'
os.chdir(ruta_raiz)

print(f"Directorio de trabajo actual: {os.getcwd()}")

### Paso 1: Función para conversión del texto base al formato necesario (PDF a Markdown)

In [ ]:
from pathlib import Path
# Importamos la función desde el archivo oculto en src/
from src.document_processing import convertir_pdf_a_markdown

# Definimos rutas relativas
ruta_entrada = Path("data/01_raw/Nombre del archivos.pdf") # Escribe aquí el nombre de tu archivo con el texto en pdf
ruta_salida = Path("data/02_interim/Nombre del archivo.md")

# Ejecutamos la función
convertir_pdf_a_markdown(ruta_pdf=ruta_entrada, ruta_md=ruta_salida)

### Paso 2: Función para clasificación maual del texto (Markdown a CSV)

In [ ]:
from pathlib import Path
from src.document_processing import convertir_pdf_a_markdown, convertir_md_a_csv

# 1. Definir nombres de archivos
nombre_archivo = "Ley de materias primas críticas"
ruta_pdf = Path(f"data/01_raw/{nombre_archivo}.pdf")
ruta_md = Path(f"data/02_interim/{nombre_archivo}.md")
ruta_csv = Path(f"data/02_interim/{nombre_archivo}_para_etiquetar.csv")

# 2. Definir las dimensiones que el usuario quiere analizar
# El usuario modifica esta lista según sus necesidades (ej. ["Similitud_MedioAmbiente", "Similitud_Economia"])
mis_dimensiones = ["Similitud_dimensión1", "Similitud_dimensión2"]

# 3. Ejecutar el flujo de transformación
print("--- INICIANDO EXTRACCIÓN DE TEXTO ---")
exito_pdf = convertir_pdf_a_markdown(ruta_pdf, ruta_md)

if exito_pdf:
    print("--- INICIANDO CREACIÓN DE PLANTILLA CSV ---")
    convertir_md_a_csv(ruta_md, ruta_csv, columnas_extra=mis_dimensiones)
    print(">>> ACCIÓN REQUERIDA: Abre el archivo CSV generado, rellena las columnas creadas con 1 y 0, y guárdalo en la carpeta 'data/03_manual/'.")

### Cálculo de embeddings para cada lista de actividades y guardado en un archivo .pkl

In [ ]:
import os
import pandas as pd
from getpass import getpass
from pathlib import Path
# Importaremos la lógica desde src/embeddings.py cuando la construyamos
from src.embeddings import calcular_embeddings_dataframe

# 4. Configuración dinámica de dimensiones y archivos
# El usuario debe mapear aquí el nombre base de sus dimensiones con el archivo .md correspondiente.
# Los archivos de entrada deben estar obligatoriamente en data/01_raw/
archivos_actividades = {
    "economia": Path("data/01_raw/actividades_dimensión1.md"), #Cambia actividades_dimensión1 por el nombre de tu archivo de actividades
    "pactoverde": Path("data/01_raw/actividades_dimensión2.md") #Cambia actividades_dimensión2 por el nombre de tu archivo de actividades
}

# 5. Autenticación segura de la API
print("--- AUTENTICACIÓN OPENAI ---")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OpenAI API Key (quedará oculta): ")

# 6. Extracción y cálculo de embeddings para las actividades
print("\n--- CALCULANDO EMBEDDINGS DE ACTIVIDADES ---")
directorio_salida = Path("data/04_embeddings")
directorio_salida.mkdir(parents=True, exist_ok=True)

for dimension, ruta_md in archivos_actividades.items():
    if not ruta_md.exists():
        print(f"✗ Error: El archivo {ruta_md} no existe. Saltando dimensión '{dimension}'.")
        continue
        
    print(f"\nProcesando dimensión: {dimension}")
    
    # Lectura y división del archivo Markdown
    with open(ruta_md, "r", encoding="utf-8") as archivo:
        contenido = archivo.read()
        
    lista_actividades = [act.strip() for act in contenido.split("\n\n") if act.strip()]
    print(f"  > Detectadas {len(lista_actividades)} actividades.")
    
    # Transformación a DataFrame para unificar el proceso de embeddings
    df_actividades = pd.DataFrame(lista_actividades, columns=["Actividad"])
    
    # Llamada a la función externa para encapsular la iteración de la API
    df_con_embeddings = calcular_embeddings_dataframe(
        df=df_actividades,
        columna_texto="Actividad",
        modelo="text-embedding-3-small"
    )
    
    # Guardado estructurado
    nombre_archivo = f"embeddings_actividades_{dimension}.pkl"
    ruta_guardado = directorio_salida / nombre_archivo
    
    df_con_embeddings.to_pickle(ruta_guardado)
    print(f"  ✓ Archivo guardado correctamente en: {ruta_guardado}")

print("\n>>> PREPARACIÓN DE DATOS FINALIZADA. Pasa al cuaderno 02_calculo_umbral.ipynb")

## Detente. Abre el archivo CSV generado en el "Paso 2". Añade las columnas requeridas, clasifica con 1 y 0, guárdalo y luego pasa al cuaderno 02.